This code detects the location of starting point of an office;

Input: List of offices + Corresponding offices 

=> Output: List of offices with coordinate information

In [1]:
import pandas as pd
import os
import cv2
import numpy as np
import json

In [2]:
def Detect_Office(Json,Office):

    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if (Office[0] == d['inferText'][0]) or (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

def Detect_Office2(Json,Office):
    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if ( d['inferText'][0] == '〇') or( d['inferText'][0] == '○') or ( d['inferText'][0] == '0')or ( d['inferText'][0] == 'O') or (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

### CLOVA FUNCTION ###
import requests
import uuid
import time
import json
import cv2
import base64

api_url = 'https://deelieyxuc.apigw.ntruss.com/custom/v1/1972/ebd01bcbf693d069817622e9839e20914143c7d0d8953eddee40e8b0af96c95b/general'
secret_key = 'S1NmVXpYZlJ0cGJ0ZEFRZXVlbkRkaHFReE9FcHNTQ0U='

def Clova(file_data):
    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(file_data).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

def Clova2(file_data):
    temp_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\Temp.jpg"
    with open(temp_path, "wb") as path:
        path.write(file_data)
    
    stream = open(temp_path, "rb")
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

    HH,WW=img.shape[:2]
    cropped=img[0:200,0:WW]
    temp_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
    cv2.imwrite(temp_path+"Temp.jpg",cropped)
    
    with open(temp_path+"Temp.jpg",'rb') as f:
        image = f.read()

    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(image).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

In [3]:
def hconcat_resize_min(im_list, interpolation=cv2.INTER_CUBIC):
    h_min = min(im.shape[0] for im in im_list)
    im_list_resize = [cv2.resize(im, (int(im.shape[1] * h_min / im.shape[0]), h_min), interpolation=interpolation)
                      for im in im_list]
    return cv2.hconcat(im_list_resize)

def Check(df):
    for n in range(1,len(df)):
        try:
            
            #Extract key info of office
            Row  = df.iloc[n]

            ###Collect Location information###
            Dept=Row["Dept"]
            Office=Row["Office"]

            print('['+str(n)+',"'+Dept+'","'+Office+'"]')
            StrPage=dta[Year][Dept][Office]['StrPage']
            StrPart=dta[Year][Dept][Office]['StrPart']
            StrLoc=int(dta[Year][Dept][Office]['StrLocation'])

            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+StrPart+".jpg",'rb')
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
            HH,WW=img.shape[:2]
            oldimg=img.copy()[0:200,0:WW]

            EndLoc=int(dta[Year][Dept][Office]['EndLocation'])
            EndPage=int(dta[Year][Dept][Office]['EndPage'])
            EndPart=dta[Year][Dept][Office]['EndPart']


            #Start#
            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+StrPart+".jpg",'rb')            
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

            Annotate=img.copy()[0:200,0:WW]
            cv2.line(Annotate,(StrLoc,0),(StrLoc,HH),(255,0,0),2)
            Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
            oldimg=hconcat_resize_min((Annotate,oldimg))

            #End#
            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(EndPage)+"\\"+EndPart+".jpg",'rb')            
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

            Annotate=img.copy()[0:200,0:WW]
            cv2.line(Annotate,(EndLoc,0),(EndLoc,HH),(255,0,0),2)
            Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
            oldimg=hconcat_resize_min((Annotate,oldimg))

            cv2.imshow(Dept+Office,oldimg)
            cv2.waitKey(0)
        except:
            print('Failed at '+Dept+Office)
    cv2.destroyAllWindows()
    cv2.waitKey(0)

In [4]:
def Show(n,Part,StrLoc):
    Row=df.iloc[n]
    Page=Row['Page']
    
    path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
    stream = open(path+"Page"+"{:03d}".format(Page)+"\\"+Part+".jpg",'rb')
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    
    HH,WW=img.shape[:2]
    cv2.line(img,(StrLoc,0),(StrLoc,HH),(225,0,0),2)
    
    cv2.imshow('Projection',img)
    cv2.waitKey(0)

#Step 1

Fill in years to refer.
Remember to keep it as a string. NOT as float.

In [215]:
Year='1942'
Showa='17'
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
os.chdir(path)
df = pd.read_csv(r'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)

In [7]:
file_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+Year+'\\DataFrame.json'
with open(file_path, encoding="utf-8") as f:
    dta = json.loads(f.read())

#Step 2

The following code will extract the location of offices.

In [8]:
n=0
FailedList=[]
#Extract key info of office
Row  = df.iloc[n]

Page=int(Row["Page"])
Office=Row["Office"]
Dept=Row['Dept']

###Insert Starting page information to motherframe###
dta[Year][Dept][Office]={}
dta[Year][Dept][Office]["Starting_Page"]=Page
print(dta[Year][Dept][Office])

###Collect Location information###
#Part1
with open("Page"+"{:03d}".format(Page)+"\\"+"Top.jpg",'rb') as f:
    file_data = f.read()
#Convert to json via CLOVA
Json=Clova(file_data)

#Find X coordinate of 'Office'.
XCoord_Unit=Detect_Office(Json,Office)
if XCoord_Unit==None:
    #Add to motherframe
    dta[str(Year)][Dept][Office]["StrLocation"]='NA'
else:
    dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
    dta[str(Year)][Dept][Office]["StrPart"]='Top'
    print(Office+ 'success: at row'+str(n))

#Part2
with open("Page"+"{:03d}".format(Page)+"\\"+"Btm.jpg",'rb') as f:
    file_data = f.read()
#Convert to json via CLOVA
Json=Clova(file_data)

#Find X coordinate of 'Office'.
XCoord_Unit=Detect_Office(Json,Office)
if XCoord_Unit==None:
    #Add to motherframe
    dta[str(Year)][Dept][Office]["StrLocation"]='NA'
    print(Office+ 'failed')
    FailedList.append(list((n,Dept,Office)))
else:
    dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
    dta[str(Year)][Dept][Office]["StrPart"]='Btm'
    print(Office+ 'success: at row'+str(n))

{'Starting_Page': 3}
秘書課success: at row0


In [9]:
#Test code| Version 2#
#Show Working office#
for n in range(1,len(df)):
    #Extract key info of office
    Row  = df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']
    print('['+str(n)+',"'+Dept+'","'+Office+'"]')
    
    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    ###Insert Starting page information to motherframe###
    dta[Year][Dept][Office]={}
    dta[Year][Dept][Office]["StrPage"]=Page
    print(dta[Year][Dept][Office])

    ###Collect Location information###
    #Part1
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Top.jpg",'rb') as f:
        file_data = f.read()
    #Convert to json via CLOVA
    try:
        Json=Clova(file_data)
    except:
        print(Office+ 'failed')
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office(Json,Office)
    print('Top Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))
    
    if XCoord_Unit!=None:
        continue
        
    #Part2
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Btm.jpg",'rb') as f:
        file_data = f.read()

    #Convert to json via CLOVA
    try:
        Json=Clova(file_data)
    except:
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office(Json,Office)
    print('Bottom Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))

[1,"総務部","文書課"]
{'StrPage': 4}
Top Loc333
文書課success: at row1
[2,"総務部","人事課"]
{'StrPage': 5}
Top Loc357
人事課success: at row2
[3,"総務部","吏務課"]
{'StrPage': 5}
Top Loc135
吏務課success: at row3
[4,"総務部","議案課"]
{'StrPage': 6}
Top LocNone
Bottom Loc415
議案課success: at row4
[5,"総務部","区政課"]
{'StrPage': 6}
Top LocNone
Bottom Loc149
区政課success: at row5
[6,"総務部","報道課"]
{'StrPage': 7}
Top Loc280
報道課success: at row6
[7,"総務部","統計課"]
{'StrPage': 7}
Top LocNone
Bottom Loc181
統計課success: at row7
[8,"企書部","企書課"]
{'StrPage': 9}
Top Loc107
企書課success: at row8
[9,"企書部","都市計書課"]
{'StrPage': 9}
Top Loc142
都市計書課success: at row9
[10,"企書部","珠算課"]
{'StrPage': 11}
Top Loc321
珠算課success: at row10
[11,"監査部","考査課"]
{'StrPage': 12}
Top Loc449
考査課success: at row11
[12,"監査部","検査課"]
{'StrPage': 12}
Top Loc449
検査課success: at row12
[13,"経理局","会計課"]
{'StrPage': 13}
Top LocNone
Bottom LocNone
会計課failed
[14,"経理局","収納課"]
{'StrPage': 14}
Top LocNone
Bottom LocNone
収納課failed
[15,"経理局","用品課"]
{'StrPage': 16}
Top Loc160
用品課success: at

#Step 3

Check for errors.
Manually type in the index, department name, and office name of erroneous department-office pair to a seperate list.

In [10]:
FailedList2=[]
for Office in FailedList:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    ###Collect Location information###
    #Part1
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Top.jpg",'rb') as f:
        file_data = f.read()
    #Convert to json via CLOVA
    try:
        Json=Clova2(file_data)
    except:
        FailedList2.append(list((n,Dept,Office)))
        continue
    
    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office2(Json,Office)
    print('Top Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))
    
    if XCoord_Unit!=None:
        continue
        
    #Part2
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Btm.jpg",'rb') as f:
        file_data = f.read()

    #Convert to json via CLOVA
    try:
        Json=Clova2(file_data)
    except:
        print(Office+ 'failed')
        FailedList2.append(list((n,Dept,Office)))
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office2(Json,Office)
    print('Bottom Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
        print(Office+ 'failed')
        FailedList2.append(list((n,Dept,Office)))
        continue
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))

Top LocNone
Bottom Loc266
会計課success: at row13
Top LocNone
Bottom Loc316
収納課success: at row14
Top LocNone
Bottom Loc167
学校営業課success: at row20
Top Loc438
装置課success: at row21
Top Loc268
町会課success: at row23
Top Loc184
動員課success: at row24
Top Loc184
貯蓄奨励課success: at row25
Top Loc124
配給第一課success: at row28
Top Loc124
配給第二課success: at row29
Top Loc233
副収入役室success: at row33
Top Loc322
学校職員課success: at row49
Top Loc381
学校施設課success: at row50
Top Loc381
初等教育課課success: at row51
Top Loc293
学校体育課success: at row54
Top LocNone
Bottom Loc177
教育研究所success: at row55
Top LocNone
Bottom Loc475
道路管理課success: at row64
Top LocNone
Bottom Loc405
防衛工事課success: at row67
Top LocNone
Bottom Loc278
築港課success: at row70
Top Loc330
飛行場建設課success: at row71
Top Loc319
経理課success: at row88
Top Loc374
車両課success: at row96
Top Loc316
電力課success: at row98
Top LocNone
Bottom Loc287
車両工場success: at row99
Top Loc23
庶務課success: at row101
Top Loc23
試験課success: at row102


In [11]:
for Office in FailedList2:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    dta[str(Year)][Dept][Office]["StrLocation"]=0
    dta[str(Year)][Dept][Office]["StrPart"]='Top'
    dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
    dta[str(Year)][ExDept][ExOffice]["EndPart"]='Top'
    dta[str(Year)][ExDept][ExOffice]["EndLocation"]=0
    dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       

In [12]:
Check(df)

[1,"総務部","文書課"]
[2,"総務部","人事課"]
[3,"総務部","吏務課"]
[4,"総務部","議案課"]
[5,"総務部","区政課"]
[6,"総務部","報道課"]
[7,"総務部","統計課"]
[8,"企書部","企書課"]
[9,"企書部","都市計書課"]
[10,"企書部","珠算課"]
[11,"監査部","考査課"]
[12,"監査部","検査課"]
[13,"経理局","会計課"]
[14,"経理局","収納課"]
[15,"経理局","用品課"]
[16,"経理局","購買課"]
[17,"経理局","地理課"]
[18,"建築部","管理課"]
[19,"建築部","営繕課"]
[20,"建築部","学校営業課"]
[21,"建築部","装置課"]
[22,"戦時生活局","庶務課"]
[23,"戦時生活局","町会課"]
[24,"戦時生活局","動員課"]
[25,"戦時生活局","貯蓄奨励課"]
[26,"戦時生活局","商工課"]
[27,"戦時生活局","農漁課"]
[28,"配給部","配給第一課"]
[29,"配給部","配給第二課"]
[30,"配給部","物価課"]
[31,"中央卸売市場","管理課"]
[32,"中央卸売市場","業務課"]
[33,"中央卸売市場","副収入役室"]
[34,"健民局","庶務課"]
[35,"健民局","軍事援護課"]
[36,"健民局","母子課"]
[37,"健民局","體力課"]
[38,"健民局","衛生課"]
[39,"健民局","防疫課"]
[40,"健民局","厚生課"]
[41,"健民局","補導課"]
[42,"公園部","管理課"]
[43,"公園部","技術課"]
[44,"養育院","庶務課"]
[45,"養育院","監護課"]
[46,"養育院","醫務課"]
[47,"養育院","会計課"]
[48,"教育局","庶務課"]
[49,"教育局","学校職員課"]
[50,"教育局","学校施設課"]
[51,"教育局","初等教育課課"]
[52,"教育局","中等教育課"]
[53,"教育局","教化課"]
[54,"教育局","学校体育課"]
[55,"教育局","教育研究所"]
[56,"教育局","日比谷図書館"]
[57,"防

Type in the department-offices with errors.

In [14]:
Type1=[[3,"総務部","吏務課"],
       [8,"企書部","企書課"],
       [9,"企書部","都市計書課"],
       [11,"監査部","考査課"],
       [12,"監査部","検査課"],
       [17,"経理局","地理課"],
       [22,"戦時生活局","庶務課"],
       [25,"戦時生活局","貯蓄奨励課"],
       [26,"戦時生活局","商工課"],
       [29,"配給部","配給第二課"],
       [33,"中央卸売市場","副収入役室"],
       [35,"健民局","軍事援護課"],
       [36,"健民局","母子課"],
       [41,"健民局","補導課"],
       [42,"公園部","管理課"],
       [46,"養育院","醫務課"],
       [48,"教育局","庶務課"],
       [49,"教育局","学校職員課"],
       [50,"教育局","学校施設課"],
       [52,"教育局","中等教育課"],
       [55,"教育局","教育研究所"],
       [56,"教育局","日比谷図書館"],
       [58,"防衛局","計書課"],
       [62,"防衛局","防火改修課"],
       [63,"土木局","庶務課"],
       [66,"土木局","治水課"],
       [68,"土木局","土木技術研究所"],
       [70,"港湾局","築港課"],
       [72,"港湾局","港務所"],
       [73,"港湾局","副収入役室"],
       [77,"水道局","給水課"],
       [80,"水道局","水原林事務所"],
       [83,"利根川水道建設所","庶務課"],
       [87,"電気局","主計課"],
       [88,"電気局","経理課"],
       [94,"事業部","運輸課"],
       [98,"事業部","電力課"],
       [102,"電気研究所","試験課"],
       [104,"清掃部","管理課"]]+FailedList2

#Step 4

See how much errors were observed in the first trial.

In [15]:
FailRate=len(Type1)/len(df)
if FailRate<0.2:
    print('Fantastic!! Success Rate is '+str(1-(len(FailedList)/len(df))))
else:
    print('残念...Success Rate is '+str(1-FailRate))
DF=pd.read_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')
DF.loc[int(Year)-1934, 'Office'] = 1-FailRate
display(DF)
DF.to_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv',index=False)

残念...Success Rate is 0.6355140186915889


,Year,Parts,WritePage,Page,PageFin,Office,OfficeFin,Unit,UnitFin,Position,PositionFin,Data
0,1934,1,1,0.985714286,.,0.9,.,NaN,.,0.08,.,.
1,1935,1,1,1,.,0.984375,.,NaN,.,-0.09375,.,.
2,1936,1,0.235294118,.,.,0.235294118,.,NaN,.,.,.,.
3,1937,1,1,1,.,0.865671642,.,NaN,.,.,.,.
4,1938,1,1,1,.,1,.,0.988095238,.,0.98816568,.,0.619957537
5,1939,1,1,1,.,0.927835052,.,1,.,.,.,.
6,1940,2,1,1,.,0.6146788990825688,.,NaN,.,.,.,.
7,1941,2,1,1,.,0.565217391,Finished!!!,NaN,.,.,.,.
8,1942,2,.,1.0,.,0.635514,.,NaN,.,.,.,.
9,1943,2,1,.,.,.,.,.,.,.,.,.


In [24]:
k=0
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=220
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

総務部 吏務課


In [27]:
k=1
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=240
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

企書部 企書課


In [32]:
k=2
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=210
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

企書部 都市計書課


In [38]:
k=3
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=400
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

監査部 考査課


In [41]:
k=4
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=200
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

監査部 検査課


In [44]:
k=5
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=120
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

経理局 地理課


In [49]:
k=6
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=310
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

戦時生活局 庶務課


In [53]:
k=7
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=140
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

戦時生活局 貯蓄奨励課


In [56]:
k=8
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=370
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

戦時生活局 商工課


In [63]:
k=9
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=410
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

配給部 配給第二課


In [67]:
k=10
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=230
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

中央卸売市場 副収入役室


In [72]:
k=11
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=240
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

健民局 軍事援護課


In [76]:
k=12
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=420
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

健民局 母子課


27

In [80]:
k=13
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=260
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

健民局 補導課


26

In [85]:
k=14
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=240
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

公園部 管理課


25

In [88]:
k=15
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=20
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

養育院 醫務課


24

In [94]:
k=16
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=260
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

教育局 庶務課


23

In [102]:
k=17
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=380
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

教育局 学校職員課


22

In [107]:
k=18
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=505
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

教育局 学校施設課


21

In [115]:
k=19
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=100
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

教育局 中等教育課


20

In [119]:
k=20
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=480
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

教育局 教育研究所


19

In [129]:
k=21
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=180
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

教育局 日比谷図書館


18

In [134]:
k=22
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=300
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

防衛局 計書課


17

In [136]:
k=23
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=440
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

防衛局 防火改修課


16

In [140]:
k=24
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=350
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

土木局 庶務課


15

In [147]:
k=25
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=340
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

土木局 治水課


14

In [149]:
k=26
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=100
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

土木局 土木技術研究所


13

In [157]:
k=27
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=420
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

港湾局 築港課


12

In [159]:
k=28
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=300
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

港湾局 港務所


11

In [161]:
k=29
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=120
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

港湾局 副収入役室


10

In [167]:
k=30
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=270
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

水道局 給水課


9

In [174]:
k=31
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=70
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

水道局 水原林事務所


8

In [179]:
k=32
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=385
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

利根川水道建設所 庶務課


7

In [183]:
k=33
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=330
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

電気局 主計課


6

In [186]:
k=34
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=30
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

電気局 経理課


5

In [190]:
k=35
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=170
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

事業部 運輸課


4

In [194]:
k=36
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=150
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

事業部 電力課


3

In [198]:
k=37
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=230
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

電気研究所 試験課


2

In [204]:
k=38
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=320
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

清掃部 管理課


1

In [206]:
for n in range(0,len(df)):
    try:
        Row=df.iloc[n]
        ExRow=df.iloc[n-1]

        Office,Dept=Row['Office'],Row['Dept']
        ExOffice,ExDept=ExRow['Office'],ExRow['Dept']    

        StrPage=Row['Page']
        ExStrPage=ExRow['Page']

        StrLoc=dta[Year][Dept][Office]['StrLocation']
        StrPart=dta[Year][Dept][Office]['StrPart']

        dta[Year][ExDept][ExOffice]['EndPage']=StrPage
        dta[Year][ExDept][ExOffice]['EndPart']=StrPart
        dta[Year][ExDept][ExOffice]['EndLocation']=StrLoc
        
        dta[Year][ExDept][ExOffice]['Page_Range']=list(range(ExStrPage,StrPage+1))

    except:
        continue

In [207]:
Check(df)

[1,"総務部","文書課"]
[2,"総務部","人事課"]
[3,"総務部","吏務課"]
[4,"総務部","議案課"]
[5,"総務部","区政課"]
[6,"総務部","報道課"]
[7,"総務部","統計課"]
[8,"企書部","企書課"]
[9,"企書部","都市計書課"]
[10,"企書部","珠算課"]
[11,"監査部","考査課"]
[12,"監査部","検査課"]
[13,"経理局","会計課"]
[14,"経理局","収納課"]
[15,"経理局","用品課"]
[16,"経理局","購買課"]
[17,"経理局","地理課"]
[18,"建築部","管理課"]
[19,"建築部","営繕課"]
[20,"建築部","学校営業課"]
[21,"建築部","装置課"]
[22,"戦時生活局","庶務課"]
[23,"戦時生活局","町会課"]
[24,"戦時生活局","動員課"]
[25,"戦時生活局","貯蓄奨励課"]
[26,"戦時生活局","商工課"]
[27,"戦時生活局","農漁課"]
[28,"配給部","配給第一課"]
[29,"配給部","配給第二課"]
[30,"配給部","物価課"]
[31,"中央卸売市場","管理課"]
[32,"中央卸売市場","業務課"]
[33,"中央卸売市場","副収入役室"]
[34,"健民局","庶務課"]
[35,"健民局","軍事援護課"]
[36,"健民局","母子課"]
[37,"健民局","體力課"]
[38,"健民局","衛生課"]
[39,"健民局","防疫課"]
[40,"健民局","厚生課"]
[41,"健民局","補導課"]
[42,"公園部","管理課"]
[43,"公園部","技術課"]
[44,"養育院","庶務課"]
[45,"養育院","監護課"]
[46,"養育院","醫務課"]
[47,"養育院","会計課"]
[48,"教育局","庶務課"]
[49,"教育局","学校職員課"]
[50,"教育局","学校施設課"]
[51,"教育局","初等教育課課"]
[52,"教育局","中等教育課"]
[53,"教育局","教化課"]
[54,"教育局","学校体育課"]
[55,"教育局","教育研究所"]
[56,"教育局","日比谷図書館"]
[57,"防

In [208]:
Error=[[29,"配給部","配給第二課"],
      [33,"中央卸売市場","副収入役室"],
      [49,"教育局","学校職員課"],
      [51,"教育局","初等教育課課"],
      [52,"教育局","中等教育課"],
      [70,"港湾局","築港課"],
      [85,"電気局","総務課"],
      [106,"清掃部","計書課"],]

In [210]:
k=0
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=400
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

配給部 配給第二課


In [218]:
k=1
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=480
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

中央卸売市場 副収入役室


In [222]:
k=2
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=385
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

教育局 学校職員課


In [227]:
k=3
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=220
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

教育局 初等教育課課


In [230]:
k=4
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=100
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

教育局 中等教育課


In [234]:
k=5
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=430
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

港湾局 築港課


In [238]:
k=6
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=320
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

電気局 総務課


In [242]:
k=7
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=390
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

清掃部 計書課


In [244]:
Check(df)

[1,"総務部","文書課"]
[2,"総務部","人事課"]
[3,"総務部","吏務課"]
[4,"総務部","議案課"]
[5,"総務部","区政課"]
[6,"総務部","報道課"]
[7,"総務部","統計課"]
[8,"企書部","企書課"]
[9,"企書部","都市計書課"]
[10,"企書部","珠算課"]
[11,"監査部","考査課"]
[12,"監査部","検査課"]
[13,"経理局","会計課"]
[14,"経理局","収納課"]
[15,"経理局","用品課"]
[16,"経理局","購買課"]
[17,"経理局","地理課"]
[18,"建築部","管理課"]
[19,"建築部","営繕課"]
[20,"建築部","学校営業課"]
[21,"建築部","装置課"]
[22,"戦時生活局","庶務課"]
[23,"戦時生活局","町会課"]
[24,"戦時生活局","動員課"]
[25,"戦時生活局","貯蓄奨励課"]
[26,"戦時生活局","商工課"]
[27,"戦時生活局","農漁課"]
[28,"配給部","配給第一課"]
[29,"配給部","配給第二課"]
[30,"配給部","物価課"]
[31,"中央卸売市場","管理課"]
[32,"中央卸売市場","業務課"]
[33,"中央卸売市場","副収入役室"]
[34,"健民局","庶務課"]
[35,"健民局","軍事援護課"]
[36,"健民局","母子課"]
[37,"健民局","體力課"]
[38,"健民局","衛生課"]
[39,"健民局","防疫課"]
[40,"健民局","厚生課"]
[41,"健民局","補導課"]
[42,"公園部","管理課"]
[43,"公園部","技術課"]
[44,"養育院","庶務課"]
[45,"養育院","監護課"]
[46,"養育院","醫務課"]
[47,"養育院","会計課"]
[48,"教育局","庶務課"]
[49,"教育局","学校職員課"]
[50,"教育局","学校施設課"]
[51,"教育局","初等教育課課"]
[52,"教育局","中等教育課"]
[53,"教育局","教化課"]
[54,"教育局","学校体育課"]
[55,"教育局","教育研究所"]
[56,"教育局","日比谷図書館"]
[57,"防

Fix the department-office pair with errors

First re-run the list of failed (pairs which was rejected), with a stricter OCR.

Open FailedList_2 and see if there are any other errors left.

If there are, fix it manually by guessing the starting x axis using 'Show(n,StrLoc)'

When you find the exact starting location, update the dataframe.

dta[Year][Dept][Office]={'Starting_Page': ,
                          'Office_X1': ,
                          'Ending_Page': ,
                          'Office_X2': ,
                          'Page_Range': []}

#Step 5

Do the same thing with the department-office pair which was wrongly accepted in the test.

#Step 6

Check the entire dataframe to confirm that the lines have been drawn at the correct place.

If there are errors, add the pair into the Type1 Error list and re-run step 5.

In [245]:
json_object = json.dumps(dta, indent=4,
                        cls=NpEncoder)
save_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+str(Year)+'\\'
with open(save_path+"DataFrame_Office.json", "w") as outfile:
    outfile.write(json_object)